# A3.7 · The agent gateway: one choke point when you scale

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.6 · Human approval that survives volume](https://spbreed.github.io/cyber-commons/lessons/A3.6.html)**.

| | |
|---|---|
| Open-source tooling | agentgateway, OPA, Keycloak |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

At one agent the controls live in the agent. At fifty, each team implements them slightly differently, none of them is audited, and the only honest answer to "is default-deny on?" is "in some of them".

## 2 · The framework

```
   one agent                     fifty agents
   +--------------+              +-----+ +-----+ +-----+ +-----+
   | controls in  |              | a1  | | a2  | | ... | | a50 |
   | the agent    |              +--+--+ +--+--+ +--+--+ +--+--+
   +--------------+                 |       |       |       |
                                    +-------+---+---+-------+
                                                v
                                        +--------------+
                                        |   gateway    | identity, policy,
                                        +------+-------+ budget, egress, log
                                               v
                                            tools

   one place to enforce, one place to audit, one place to turn off
```

**Mitigates: every threat in this chapter, at one enforcement point.**

Everything in Chapters 2 and 3 works. The problem is where it lives.

At one agent, the controls sit in the agent, and that is fine. At fifty, it
stops being fine for reasons that have nothing to do with security engineering:

- Each team implements provenance, budgets and egress slightly differently.
- Nobody can answer "is this control on, everywhere" without reading fifty
  repositories.
- A new agent starts at zero and re-earns every control by hand.
- Fixing a control means fifty pull requests and a migration.

The **gateway** is the same controls, moved to a point every call must pass
through. It holds identity (A2.1–A2.3), policy (A3.1), egress (A3.3), budgets
(A3.4) and audit (A2.7). An agent that implements none of them still gets all of
them, because the enforcement is no longer the agent's responsibility.

It also solves a problem nothing else does: **downstream systems that cannot
consume a delegated identity.** A legacy database or a vendor API that only
understands a static credential forces that credential back into agent code —
undoing A2.3 completely. The gateway holds it instead, authorises the *user*
before the call, and presents the static credential onward. The agent never sees
it.

The honest cost: the gateway is now a single point of failure and a very
attractive target. It has to be operated accordingly.

> **What this control closes.**
>
> Not a new control. The same controls, at a point every call passes through — and the only answer to a downstream that cannot consume delegated identity.

## 3 · The control

In [ ]:
LEGACY_DB_CREDENTIAL = "static-service-password"     # never leaves the gateway

REGISTRY = {"spiffe://corp/reports-agent": {"owner": "sam@corp", "expires": 9000}}
POLICY = {("reports-agent", "run_query", "table:reports"): {"SELECT"}}
EGRESS_ALLOW = {"reports-db.corp.example"}

AUDIT = []

def gateway(call):
    """One choke point: identity, registry, policy, egress, budget, audit."""
    checks = []
    def check(name, ok, why=""):
        checks.append((name, ok, why)); return ok

    if not check("identity", call["identity"] in REGISTRY, "attested and registered"):
        return {"allowed": False, "checks": checks}
    verbs = POLICY.get((call["agent"], call["tool"], call["resource"]), set())
    if not check("policy", call["verb"] in verbs, f"permitted verbs {sorted(verbs) or 'none'}"):
        return {"allowed": False, "checks": checks}
    if not check("egress", call["destination"] in EGRESS_ALLOW, "destination allow-list"):
        return {"allowed": False, "checks": checks}
    if not check("budget", call["calls_so_far"] < 5, "per-target ceiling"):
        return {"allowed": False, "checks": checks}

    # the agent never held this; the gateway attaches it on the way out
    AUDIT.append({"principal": call["principal"], "agent": call["agent"],
                  "tool": call["tool"], "resource": call["resource"]})
    return {"allowed": True, "checks": checks, "credential_attached": LEGACY_DB_CREDENTIAL[:6] + "..."}

BASE = {"identity": "spiffe://corp/reports-agent", "agent": "reports-agent",
        "principal": "dana@corp", "tool": "run_query", "resource": "table:reports",
        "verb": "SELECT", "destination": "reports-db.corp.example", "calls_so_far": 0}

CASES = {
 "the intended call":          BASE,
 "unregistered agent":         dict(BASE, identity="spiffe://corp/rogue-agent"),
 "verb not permitted":         dict(BASE, verb="DELETE"),
 "exfiltration destination":   dict(BASE, destination="archive.evil.example"),
 "over the per-target ceiling":dict(BASE, calls_so_far=9),
}
for label, call in CASES.items():
    r = gateway(call)
    failed = [n for n, ok, _ in r["checks"] if not ok]
    print(f"   {label:28s}{'ALLOWED' if r['allowed'] else 'denied at ' + failed[0]}")

print(f"\naudit entries written: {len(AUDIT)}")
print(f"credential held by the agent: never - attached at the gateway")
print()
print("The agent implements none of this. Add a new agent tomorrow and it")
print("inherits every control by being on the other side of one hop.")
print()
print("And the legacy database, which cannot consume a delegated token, is")
print("reached with a static credential the agent has never seen - authorised")
print("against dana before the call was made.")
assert len(AUDIT) == 1 and gateway(CASES["unregistered agent"])["allowed"] is False

## What you just proved

Five calls hit one gateway. The intended call is allowed and audited with the human principal attached; the unregistered agent, the unpermitted verb, the exfiltration destination and the over-budget call are each denied at the first check that catches them — and the legacy credential is attached at the gateway, never held by the agent.

## Your turn

Count your agents. If it is more than five, work out how you would currently answer 'is egress control on for all of them' — and how long that would take.

## Where this leaves you

**What you can do now.** Four independent layers now stand between a compromised agent and a consequence — the policy decision at the tool call, the sandbox, the egress boundary and the budget — and at scale they collapse into one gateway you can audit and switch off.

**What you still cannot do.** You have a secured architecture and nothing that builds on it. Every control here is stated as a rule; none of it is a pipeline anyone operates, and the first agentic system most organisations run is a security tool that reads untrusted code all day.

**Function B builds that system as an SDLC, and holds it to every rule in this chapter. Next → B1.0, what an AI SDLC means.**

---

**Next → [B1.0 · Start here — what an AI SDLC means](https://spbreed.github.io/cyber-commons/lessons/B1.0.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*